# Vilex — Kaggle Full Pipeline (Vertex AI + OmniVoice)

**Repo:** https://github.com/thaiphu05/Vilex  
**Model:** `gemini-3.6-flash` (Vertex AI via `GEMINI_CREDENTIALS`)  
**Pipeline:** Stage 1 speechify → 1.5 slot dictation → 1.75 disfluency → Stage 4 synthesis → 4b backchannel → Stage 5 OmniVoice TTS

All parameters live in **`config.yaml`**; this notebook overrides them through `VILEX_*` env vars (dotted key with `__`, e.g. `VILEX_RUN__DATASETS`). See `docs/CONFIGURATION.md`.

> ⚠️ **2-phase execution required** — Kaggle has one interpreter. Stages 1–4 need `transformers>=4.53`; Stage 5 (OmniVoice) conflicts with the vendored Chatterbox pin `4.46.3`. Run Phase 1, then **Kernel → Restart & Clear Output**, then Phase 2.

## Kaggle Settings (do once)
- `Settings → Internet ON`
- `Settings → Accelerator → GPU T4 x2` (Stage 5 GPU; Stages 1–4 can run CPU)
- `Add-ons → Secrets` → `GEMINI_CREDENTIALS_JSON` = service-account JSON; optional `GEMINI_LOCATION`
- Upload `voice_clone/` as a Kaggle Dataset (each `*.wav` needs a sidecar `*.txt`)

---
## PHASE 1 — Stages 1–4b (Dialogue Generation)
Run top-to-bottom, then **Restart kernel** before Phase 2.

In [ ]:
# Cell 1 — Clone repo
!git clone https://github.com/thaiphu05/Vilex.git
%cd Vilex
!pwd && git log --oneline -3 && ls -lh

In [ ]:
# Cell 2 — Install Stages 1–4 deps
!pip install -q -r requirements.txt
!pip install -q google-genai google-auth
!python -c "import transformers, torch, yaml; print(f'transformers={transformers.__version__} torch={torch.__version__} cuda={torch.cuda.is_available()}')"

In [ ]:
# Vertex AI auth (service-account JSON from Kaggle Secrets).
import os, json, base64
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
sa_json = secrets.get_secret("GEMINI_CREDENTIALS_JSON")
if not sa_json or not sa_json.strip():
    raise SystemExit("Missing Kaggle Secret GEMINI_CREDENTIALS_JSON")

sa_text = sa_json.strip()
if not sa_text.lstrip().startswith("{"):
    try:
        sa_text = base64.b64decode(sa_text).decode("utf-8")
    except Exception:
        pass

sa_path = Path("sa.json").resolve()
sa_path.write_text(sa_text, encoding="utf-8")
os.environ["GEMINI_CREDENTIALS"] = str(sa_path)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(sa_path)

try:
    loc = secrets.get_secret("GEMINI_LOCATION")
except Exception:
    loc = None
os.environ["GEMINI_LOCATION"] = (loc.strip() if loc and loc.strip() else "global")

info = json.loads(sa_text)
print(f"SA project: {info.get('project_id')}  location: {os.environ['GEMINI_LOCATION']}")

In [ ]:
# Pipeline config — overrides config.yaml via VILEX_* env vars (see docs/CONFIGURATION.md).
import os, json

MODEL = "gemini-3.6-flash"           # writer / boundary / tt / bc model
DATASETS = ["interviewer"]           # e.g. ["interviewer", "multiwoz", "negotiator"]
SPLITS = ["train"]                   # ["train", "test"]

def set_vi(overrides):
    """Set VILEX_* env vars consumed by src/config.py (dotted key -> VILEX_A__B)."""
    for key, val in overrides.items():
        if isinstance(val, bool):
            sval = "true" if val else "false"
        elif isinstance(val, (list, dict)):
            sval = json.dumps(val, ensure_ascii=False)
        else:
            sval = str(val)
        os.environ["VILEX_" + key.upper().replace(".", "__")] = sval

set_vi({
    "run.datasets": DATASETS,
    "run.splits": SPLITS,
    "paths.results_root": "data/results_vi",
    "paths.results_xt_root": "data/results_vi_xt",
    "paths.results_dis_root": "data/results_vi_dis",
    "paths.synthesis_root": "data/vi_tt",
    "paths.bc_root": "data/vi_tt_bc",
    "paths.audio_root": "data/vi_audio",
    "llm.writer_model": MODEL,
    "llm.boundary_model": MODEL,
    "llm.tt_model": MODEL,
    "llm.bc_model": MODEL,
    "stage1_speechify.max_train_samples": 1,   # 0 = all
    "stage1_speechify.max_test_samples": 1,
    "stage4_synthesis.max_dialogues": 1,       # 0 = all
    "stage4_synthesis.max_turns": 0,           # 0 = auto (len(source_turns))
})
print("VILEX overrides set:", DATASETS, SPLITS)

In [ ]:
# Cell 4 — Stage 1: Speechify (clean -> spoken VI)
!python -m src.speechify_run

In [ ]:
# Cell 5 — Stage 1.5: Cross-turn slot dictation
!python -m src.cross_turn_slots

In [ ]:
# Cell 6 — Stage 1.75: Disfluency injection (+ "..." -> [PAUSE])
!python -m src.disfluency

In [ ]:
# Cell 7 — Stage 4: Synthesis (slot detection + turn-taking)
!python -m src.synthesis.run

In [ ]:
# Cell 8 — Stage 4b: Backchannel text
!python -m src.synthesis.run_add_bc

In [ ]:
# Cell 9 — Verify Phase 1 outputs
!echo "=== data/results_vi ===" && find data/results_vi -type f | head -20
!echo "=== data/vi_tt ===" && find data/vi_tt -type f | head -20
!echo "=== data/vi_tt_bc ===" && find data/vi_tt_bc -type f | head -20

### ⏸️ STOP — before Phase 2
1. **Save outputs**: `Save Version` (persists `outputs/` as Output), or keep them in `/kaggle/working` if you just restart.
2. **Kernel → Restart & Clear Output** (required: clears transformers 4.53 so Stage 5 can install its deps).
3. After restart, run Phase 2. If you saved outputs as a Dataset, mount it via `Settings → Add Input`.

---
## PHASE 2 — Stage 5 (OmniVoice TTS)
Run **after kernel restart**.

In [ ]:
# Cell 10 — Re-enter repo + verify Phase 1 outputs still exist
import os
from pathlib import Path
for cand in ["/kaggle/working/Vilex", "Vilex", "/kaggle/working"]:
    if Path(cand).is_dir() and (Path(cand) / "tts_render").is_dir():
        os.chdir(cand)
        break
print(f"CWD={os.getcwd()}")
!pwd && ls -lh
!find outputs -type f 2>/dev/null | head -20

In [ ]:
# Cell 11 — Install Stage 5 deps (OmniVoice + Qwen3 forced aligner)
!pip install -q -r requirements-stage5.txt
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
# OmniVoice may pull an older transformers; re-assert the aligner requirement.
!pip install -q 'transformers>=4.56'
!python -c "import torch; print(f'torch={torch.__version__} cuda={torch.cuda.is_available()}')"
!python -c "import silero_vad, yaml; from transformers import AutoProcessor, AutoModelForTokenClassification; print('silero+transformers+yaml ok')"

In [ ]:
# Stage 5 config — discover input dialogues + voice pool, then override config.yaml.
import os, json, glob

VOICE_POOL = "/kaggle/input/voice-clone"   # <- your voice-clone dataset mount
for cand in [VOICE_POOL, "/kaggle/input/voice_clone", "voice_clone",
             "/kaggle/working/voice_clone"]:
    if os.path.isdir(cand):
        VOICE_POOL = cand
        break

# Stage-4b output: local data/vi_tt_bc if it persisted, else a Kaggle Dataset mount.
BC_ROOT = "data/vi_tt_bc"
if not glob.glob(os.path.join(BC_ROOT, "**", "text_dialogue_*", "*", "*.json"), recursive=True):
    for root in sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(root, "**", "text_dialogue_*", "*", "*.json"), recursive=True):
            BC_ROOT = root
            break

def set_vi(overrides):
    for key, val in overrides.items():
        if isinstance(val, bool):
            sval = "true" if val else "false"
        elif isinstance(val, (list, dict)):
            sval = json.dumps(val, ensure_ascii=False)
        else:
            sval = str(val)
        os.environ["VILEX_" + key.upper().replace(".", "__")] = sval

set_vi({
    "paths.bc_root": BC_ROOT,               # recursive glob: any depth under it
    "paths.audio_root": "data/vi_audio",
    "paths.voice_clone_pool": VOICE_POOL,
    "stage5_tts.backend": "omnivoice",
    "stage5_tts.language": "vi",
    "stage5_tts.device": "cuda",            # OOM -> "cpu"
    "stage5_tts.num_variants": 1,
    "stage5_tts.max_dialogues": 1,          # 0 = all
    "stage5_tts.tags.render": True,         # keep [laughter]/[sigh]/... tags
})

wavs = sorted(glob.glob(os.path.join(VOICE_POOL, "*.wav")))
missing = [w for w in wavs if not os.path.isfile(os.path.splitext(w)[0] + ".txt")]
print(f"VOICE_POOL={VOICE_POOL} ({len(wavs)} wavs, {len(missing)} missing .txt)")
print(f"BC_ROOT={BC_ROOT}")
if len(wavs) < 2:
    print("ERROR: voice pool needs >=2 wavs with sidecar .txt")

In [ ]:
# Cell 12 — Stage 5: OmniVoice render
!python tts_render/convert_spoken.py

In [ ]:
# Preview + export
!find data/vi_audio -type f 2>/dev/null | head -30
try:
    from IPython.display import Audio, display
    import glob as _g
    wavs = _g.glob("data/vi_audio/**/dialogue.wav", recursive=True)
    if wavs:
        print(f"Preview: {wavs[0]}")
        display(Audio(wavs[0]))
    else:
        print("No dialogue.wav yet — check logs above")
except Exception as e:
    print(e)
!zip -qr /kaggle/working/vi_audio.zip data/vi_audio
!ls -lh /kaggle/working/vi_audio.zip 2>/dev/null

## Notes
- **Full run:** set `stage1_speechify.max_train_samples: 0`, `stage4_synthesis.max_dialogues: 0`, `stage5_tts.max_dialogues: 0` in the config cells (or edit `config.yaml`). LLM calls are rate-limited (`llm.gemini_min_interval`), so a full dataset takes hours.
- **One-command alternative:** `./run_vi_pipeline.sh` runs the same stages with no flags.
- **Override anything without editing files:** `VILEX_RUN__DATASETS='["interviewer","soda"]'` etc. — see `docs/CONFIGURATION.md`.
- **Output layout:** `data/vi_audio/text_dialogue_<ds>/<split>/<id>/varNN/dialogues/dialogue.wav` (stereo, ch0=assistant ch1=user) + `meta.json`.